In [ ]:
##########################
## ÚLTIMA VERSÃO 13042026
##########################


!pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##################################################
##################################################
###########  PARA CADA MÊS A SER ATUALIZADO
###########  FORNEÇA MÊS e ANO
##################################################
##################################################

mes = "02"
ano = "2026"

anomes = f"{ano}{mes}"
Caminho="/content/drive/My Drive/PNAD/CAGED"
dados = f"CAGEDMOV{anomes}"

In [ ]:
# importa o arquivo do IBGE para o mes/ano fornecidos e grava no Drive

!wget -O "{Caminho}/{dados}.7z" \
"ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/{ano}/{anomes}/{dados}.7z"  # IBGE

# Descompacta o arquivo baixado para .txt
!apt-get install -y p7zip-full
!7z x "{Caminho}/{dados}.7z" -o"{Caminho}"

# Apaga o arquivo zipado do Drive
!rm "{Caminho}/{dados}.7z"

--2026-03-25 14:00:25--  ftp://ftp.mtps.gov.br/pdet/microdados/NOVO%20CAGED/2026/202601/CAGEDMOV202601.7z
           => ‘/content/drive/My Drive/PNAD/CAGED/CAGEDMOV202601.7z’
Resolving ftp.mtps.gov.br (ftp.mtps.gov.br)... 189.9.32.26
Connecting to ftp.mtps.gov.br (ftp.mtps.gov.br)|189.9.32.26|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /pdet/microdados/NOVO CAGED/2026/202601 ... done.
==> SIZE CAGEDMOV202601.7z ... 52965302
==> PASV ... done.    ==> RETR CAGEDMOV202601.7z ... done.
Length: 52965302 (51M) (unauthoritative)

CAGEDMOV202601.7z   100%[===================>]  50.51M   888KB/s    in 59s     

2026-03-25 14:01:26 (879 KB/s) - ‘/content/drive/My Drive/PNAD/CAGED/CAGEDMOV202601.7z’ saved [52965302]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove 

In [ ]:
# CRIAR BASE PORTO FELIZ
PortoFeliz = pd.read_csv(f"{Caminho}/{dados}.txt", sep=";", decimal=",", header=0)

# Converter a coluna para inteiro (se necessário)
PortoFeliz["município"] = PortoFeliz["município"].astype(int)

# 2. Aplicar os filtros
PortoFeliz = PortoFeliz[
    (PortoFeliz["município"] == 354060) &
    (PortoFeliz["categoria"] != "111") &
    (PortoFeliz["seção"] == "C") &
    (PortoFeliz["salário"] > 396) &
    (PortoFeliz["salário"] < 30000) &
    (PortoFeliz["salário"] != 0.00)
]

# Criando estatísticas para base Porto Feliz

# Agrupar e calcular média de salário
media_salario = (
    PortoFeliz
    .groupby(["competênciamov", "saldomovimentação"], as_index=False)
    .agg(X=("salário", "mean"))
)

# Exportar para Excel
media_salario.to_excel(
    f"{Caminho}/PORTO/{anomes}_media.xlsx",
    index=False)

# Agrupar e contar número de registros
num_admitidos = (
     PortoFeliz
    .groupby(["competênciamov", "saldomovimentação"], as_index=False)
    .agg(n=("salário", "size"))   # ou qualquer coluna que sempre exista
)

# num_admitidos = (
#     PortoFeliz
#    .groupby(["competênciamov", "saldomovimentação"], as_index=False)
#    .size())

# Exportar para Excel
num_admitidos.to_excel(
    f"{Caminho}/PORTO/{anomes}_NUMADMITIDO.xlsx",
    index=False)

# Selecionandos os CBOs de interesses - Ocupações de interesse

codigos = [
    725505, 784205, 724315, 721210, 391205, 721215, 414105, 724410, 911305, 862150,
    721405, 721430, 411010, 724325, 391125, 391210, 414125, 411005, 414140, 782220,
    811770, 720210, 951105, 725205, 391115, 723330, 821450, 354205, 725015, 252405,
    141205, 722320, 725010, 721105, 252105, 724310, 723315, 314110, 391105, 782130,
    351605, 318610, 821405, 252210, 410105, 724510, 725105, 781110, 354125, 721225,
    721410, 782510, 722205, 414110, 214905, 720215, 821315, 722230, 395105, 413110,
    715615, 821435, 414210, 724205, 724220, 773345, 724440, 391135, 721220, 391130,
    317115, 720150, 821215, 313120, 214405, 723125, 413115, 142105, 722215, 722315,
    311105, 722115, 142605, 391120, 950110, 313215, 391215, 313405, 821305, 203220,
    721325, 822120, 821205, 314410, 314205, 342125, 214910, 715220, 300305, 121010,
    142305, 142205, 311410, 354305, 414135, 723120, 782115, 723215, 203210, 822115,
    410205, 821220, 142320, 721115, 142605, 391120, 950110, 313215, 391215, 313405,
    822120, 821205, 314410, 314205, 342125, 214910, 715220, 300305, 121010, 142305,
    142205, 311410, 354305, 414135, 723120, 782115, 723215, 203210, 822115, 410205,
    821220, 142320, 721115, 782305, 720110, 352305, 722225, 301105, 212405, 142335,
    212410, 514310, 141615, 233210, 711230, 322215, 351305, 514325, 721315, 318205,
    730105, 715315, 142705, 142115, 723225, 782310, 354210, 252545, 318605, 721425,
    313205, 311515, 318005, 300105, 212420, 351115, 763210, 142405, 954125, 950305,
    520110, 354135, 313130, 411030, 724405, 414215, 214420, 911325, 725420, 225140,
    723320, 860110, 773355, 821445, 910105, 253215, 142315, 720125, 821440, 910130,
    142330, 214315, 214610, 724515, 422105, 410240, 314305, 252705, 724110, 142210,
    913115, 811760, 951315, 725310, 252305, 724415, 313115, 214915, 919105, 413105,
    123105, 214305, 722310, 723220, 725225, 766145, 252310, 314710, 725020, 731165,
    318015, 715605, 724435, 913120, 720135, 773315, 721415, 724505, 840105, 312105,
    821305, 203220, 721325, 142535, 142520, 142410, 722210, 722305, 820110, 820105,
    717020, 410235, 314125, 352205, 783225,
    342105, 318510, 722105, 820125, 731135, 731145, 721205, 720160, 314715, 314720,
    314105, 314620, 214615, 212310, 253120, 992110, 318420, 214605, 722410, 822125,
    142505, 911315, 720140, 771110, 722220, 311715, 317110, 724320, 410230, 311505,
    314730, 712225, 725005, 252715, 724105, 818105, 123305, 223530, 141305, 771105,
    914405, 911110, 725705, 811105, 731160, 724210, 773205, 715135, 725510, 342110,
    521115, 510310, 351105, 202120, 212315, 252515, 252720, 414205, 715305, 818110,
    411045, 342305, 318010, 262420, 123110, 123705, 122205, 715610, 954115, 313105,
    214240, 201215, 991305, 142415, 142510, 141605, 252605, 992120, 514315, 919305,
    720220, 720145, 774105, 862120, 317205, 722325, 782110, 811110, 762325, 841115,
    813125, 711245, 723235, 203225, 773125, 342315, 410215, 910115, 520105, 317210,
    354140, 314615, 351505, 314115, 314210, 311615, 422205, 823325, 722415, 351315,
    354120, 253115, 261215, 212415, 513435, 252205, 411035, 731170, 722405, 777210,
    781105, 241040, 412205, 513425, 262410, 318505, 318710, 318310, 318110, 123405,
    123115, 953110, 512105, 992205, 214310, 214330, 214505, 214530, 201210, 201220,
    823210, 721110, 722110, 142310, 142110, 142530, 766205, 724115, 914410, 720115,
    720130, 810110, 731125, 766315, 821245, 813110, 715115, 766320, 862505, 731175,
    731180, 761240, 711215, 641010, 715125, 715210, 203105, 723325, 142325, 773120,
    773130, 740105, 410220, 820210, 860105, 420135, 768110, 313125, 313210, 314610,
    313505, 316110, 314120, 214365, 214435, 142120, 214930, 214935,
    841505, 848210, 848215, 848205               # Novos CBOs - Industria leiteira
]

# Criando estatísticas para Porto Feliz (com filtro de OCUPAÇÃO)

PortoFeliz_Ocupados = PortoFeliz[
    PortoFeliz["cbo2002ocupação"].isin(codigos)
]

# Agrupar e calcular salário médio
media_salario = (
    PortoFeliz_Ocupados
    .groupby(["competênciamov", "saldomovimentação"], as_index=False)
    .agg(X=("salário", "mean")))

# Exportar para Excel
media_salario.to_excel(
    f"{Caminho}/PORTO_OCUPADOS/{anomes}_media.xlsx",
    index=False)

# Agrupar e contar número de registros
num_admitidos = (
    PortoFeliz_Ocupados
    .groupby(["competênciamov", "saldomovimentação"], as_index=False)
    .size())

# Renomear a coluna resultante
num_admitidos = num_admitidos.rename(columns={"size": "n"})

# Exportar para Excel
num_admitidos.to_excel(
    f"{Caminho}/PORTO_OCUPADOS/{anomes}_NUMADMITIDO.xlsx",
    index=False)

In [ ]:
# AQUI TERMINA A ATULIZAÇÃO DOS MESES E COMEÇA A UNIFICAÇÃO DOS DADOS.

In [ ]:
import os
import glob

# Craindo diretórios
d2 = "/content/drive/My Drive/PNAD/CAGED/PORTO"
d3 = "/content/drive/My Drive/PNAD/CAGED/PORTO_OCUPADOS"

# Função auxiliar para ler todos os arquivos de um padrão
def ler_arquivos(diretorio, padrao):
    arquivos = glob.glob(os.path.join(diretorio, padrao))
    lista_df = [pd.read_excel(arq) for arq in arquivos]
    return pd.concat(lista_df, ignore_index=True)

# Lendo e unificando os arquivos
dados_MEDIA_PORTO         = ler_arquivos(d2, "*_media.xlsx")
dados_ADMITIDOS_PORTO     = ler_arquivos(d2, "*_NUMADMITIDO.xlsx")
dados_MEDIA_PORTO_OCUPADOS     = ler_arquivos(d3, "*_media.xlsx")
dados_ADMITIDOS_PORTO_OCUPADOS = ler_arquivos(d3, "*_NUMADMITIDO.xlsx")

In [ ]:
# Leitura do arquivo do INPC atualizado - último disponível.

import requests
import re
import json
import csv
import os
from collections import defaultdict

def download_inpc_csv():
    url = "https://www.dadosdemercado.com.br/indices/inpc"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    print(f"Acessando {url}...")
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Erro ao acessar a página: {response.status_code}")
        return

    html_content = response.text
    match = re.search(r'const data = (\[\[.*?\]\]);', html_content, re.DOTALL)

    if not match:
        print("Não foi possível encontrar os dados no HTML.")
        return

    data_str = match.group(1)
    try:
        raw_data = json.loads(data_str)
    except json.JSONDecodeError as e:
        print(f"Erro ao decodificar os dados: {e}")
        return

    # Reorganizar os dados para o formato de matriz (Ano x Mês)
    processed_data = defaultdict(lambda: {str(i).zfill(2): '' for i in range(1, 13)})
    years = set()

    for item in raw_data:
        date_str = item[0]
        value = item[1]

        year = date_str[:4]
        month = date_str[5:7]

        processed_data[year][month] = str(value).replace('.', ',') + '%'
        years.add(year)

    sorted_years = sorted(list(years), reverse=True)
    months_order = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
    month_names = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

    # Salvar como CSV no caminho especificado
    save_directory = "/content/drive/My Drive/PNAD/CAGED"
    os.makedirs(save_directory, exist_ok=True)

    filename = os.path.join(save_directory, "indices.csv")
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # Escrever cabeçalho
        header = ["Ano"] + month_names + ["Ano"]
        writer.writerow(header)

        # Escrever dados
        for year in sorted_years:
            row = [year]
            for month in months_order:
                row.append(processed_data[year].get(month, '--')) # Usar '--' para meses sem dados

            # Calcular a soma anual se houver dados para o ano
            annual_sum = 0.0
            has_data_for_year = False
            for month_val in processed_data[year].values():
                if month_val and month_val != '--':
                    try:
                        # Remover '%' e substituir ',' por '.' para conversão
                        clean_val = month_val.replace('%', '').replace(',', '.')
                        annual_sum += float(clean_val)
                        has_data_for_year = True
                    except ValueError:
                        pass # Ignorar valores não numéricos

            if has_data_for_year:
                row.append(f"{annual_sum:.2f}%".replace('.', ','))
            else:
                row.append('--')

            writer.writerow(row)

    print(f"Arquivo '{filename}' salvo com sucesso!")
    print(f"Total de anos processados: {len(sorted_years)}")

if __name__ == "__main__":
    download_inpc_csv()

Acessando https://www.dadosdemercado.com.br/indices/inpc...
Arquivo '/content/drive/My Drive/PNAD/CAGED/indices.csv' salvo com sucesso!
Total de anos processados: 27


In [ ]:
#### ORGANIZANDO O ARQUIVO DE INPC PARA SE JUNTAS AS BASE DE MÉDIAS (SÃO 3 BASES)

INPC = pd.read_csv("/content/drive/My Drive/PNAD/CAGED/indices.csv")

# Transformar de wide para long (Jan-Dez viram uma coluna 'mes')
inpc = INPC.melt(
    id_vars=["Ano"],
    value_vars=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],
    var_name="mes",
    value_name="inpc")

# Renomear categorias de mês para números
mapa_meses = {
    "Jan":"01","Fev":"02","Mar":"03","Abr":"04","Mai":"05","Jun":"06",
    "Jul":"07","Ago":"08","Set":"09","Out":"10","Nov":"11","Dez":"12"}
inpc["mes"] = inpc["mes"].map(mapa_meses)

# 5. Concatenar ano+mes
inpc["Anomes"] = inpc["Ano"].astype(str) + inpc["mes"]

# 6. Selecionar apenas colunas desejadas
inpc = inpc[["Anomes", "inpc"]]

In [ ]:
########################################################
#### Preparando os bancos de MÉDIAS para agregar o INPC
########################################################

# Renomeando colunas
dados_MEDIA_PORTO = dados_MEDIA_PORTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_PORTO_OCUPADOS = dados_MEDIA_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

# Ajustando categorias da coluna 'contrato'
mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}

dados_MEDIA_PORTO["contrato"] = dados_MEDIA_PORTO["contrato"].map(mapa_contrato)
dados_MEDIA_PORTO_OCUPADOS["contrato"] = dados_MEDIA_PORTO_OCUPADOS["contrato"].map(mapa_contrato)

In [ ]:
###############################################################
#### Organizando os bancos de Número de admitidos para plotagem
###############################################################

# Renomeando colunas
dados_ADMITIDOS_PORTO = dados_ADMITIDOS_PORTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n"  })

dados_ADMITIDOS_PORTO_OCUPADOS = dados_ADMITIDOS_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })

# Ajustando categorias da coluna 'contrato'
mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}

dados_ADMITIDOS_PORTO["contrato"] = dados_ADMITIDOS_PORTO["contrato"].map(mapa_contrato)
dados_ADMITIDOS_PORTO_OCUPADOS["contrato"] = dados_ADMITIDOS_PORTO_OCUPADOS["contrato"].map(mapa_contrato)

In [ ]:
### AGREGANDO INPC NAS BASES DE MÉDIAS por ano/mes
################################################################################

# Garantir que 'Anomes' seja string em todas as bases
dados_MEDIA_PORTO["Anomes"] = dados_MEDIA_PORTO["Anomes"].astype(str)
dados_MEDIA_PORTO_OCUPADOS["Anomes"] = dados_MEDIA_PORTO_OCUPADOS["Anomes"].astype(str)

inpc["Anomes"] = inpc["Anomes"].astype(str)

# Merge com INPC
dados_MEDIA_PORTO = pd.merge(dados_MEDIA_PORTO, inpc, on="Anomes")
dados_MEDIA_PORTO_OCUPADOS = pd.merge(dados_MEDIA_PORTO_OCUPADOS, inpc, on="Anomes")

# Função auxiliar para limpar INPC
def limpar_inpc(df):
    df["inpc"] = (
        df["inpc"]
        .astype(str)                # garante que é string
        .str.replace(",", ".", regex=False)  # troca vírgula por ponto
        .str.replace("%", "", regex=False)   # remove símbolo de porcentagem
    )
    df["inpc"] = pd.to_numeric(df["inpc"], errors="coerce")  # converte para numérico
    return df

# 3. Aplicar limpeza em cada base
dados_MEDIA_PORTO = limpar_inpc(dados_MEDIA_PORTO)
dados_MEDIA_PORTO_OCUPADOS = limpar_inpc(dados_MEDIA_PORTO_OCUPADOS)

In [ ]:
#### AGREGANDO INPC ACUMULADO e DEFLACIONAMENTO
####  Porto Feliz E ocupados
#################################################

def calcular_inpc(df):
    # 1. Converter INPC em fator (1 + inpc/100)
    df["inpc_fator"] = 1 + df["inpc"] / 100

    # 2. Calcular INPC acumulado (produto cumulativo)
    df["inpc_acm"] = df["inpc_fator"].cumprod()

    # 3. Calcular salário deflacionado
    df["Sal_def_INPC"] = df["Media_Salario"] / df["inpc_acm"]

    return df

# Aplicar para cada base
dados_MEDIA_PORTO = calcular_inpc(dados_MEDIA_PORTO)
dados_MEDIA_PORTO_OCUPADOS = calcular_inpc(dados_MEDIA_PORTO_OCUPADOS)

In [ ]:
###  GRÁFICO 3 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO["Anomes"],
    "categoria": dados_MEDIA_PORTO["contrato"],
    "inpc": dados_MEDIA_PORTO["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP3_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP3_inpc.index = pd.to_datetime(SP3_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP3_inpc, x=SP3_inpc.index, y=SP3_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (sem filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 4 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO["contrato"],
    "n": dados_ADMITIDOS_PORTO["n"]
})

# Converter de long para wide
SP4_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP4_n.index = pd.to_datetime(SP4_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP4_n, x=SP4_n.index, y=SP4_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (sem filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 5 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO_OCUPADOS["Anomes"],
    "categoria": dados_MEDIA_PORTO_OCUPADOS["contrato"],
    "inpc": dados_MEDIA_PORTO_OCUPADOS["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP5_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP5_inpc.index = pd.to_datetime(SP5_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP5_inpc, x=SP5_inpc.index, y=SP5_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (com filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 6 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO_OCUPADOS["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO_OCUPADOS["contrato"],
    "n": dados_ADMITIDOS_PORTO_OCUPADOS["n"]
})

# Converter de long para wide
SP6_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP6_n.index = pd.to_datetime(SP6_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP6_n, x=SP6_n.index, y=SP6_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (com filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_Admitidos.html", include_plotlyjs="cdn")

fig.show()